Bringing over the same set up as my base model (which will be ran last for comparison)

In [ ]:
import pickle
import pandas as pd

df = pd.read_pickle('../data/cleaned_games.pkl')

with open('../data/genre_columns.pkl', 'rb') as f:
    genre_features = pickle.load(f)

features = [
    'price',
    'year',
    'num_tags',
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs',
] + list(genre_features)

X = df[features]
y = df['hit']

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]



Accuracy (Model Base): 0.9287656017070532
F1 Score (Model Base): 0.3019192208536236
              precision    recall  f1-score   support

           0       0.98      0.94      0.96     33069
           1       0.22      0.46      0.30      1142

    accuracy                           0.93     34211
   macro avg       0.60      0.70      0.63     34211
weighted avg       0.96      0.93      0.94     34211

[[31247  1822]
 [  615   527]]


Now, with this as a basis, I will complete some model analysis before moving into my second layer, the main purpose of this project. I will start by separating th efeatures into subgroups to see if any one group is entirely carrying the model.

In [ ]:
dev_features = [
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs'
]

meta_features = ['num_tags']

genre_features = list(genre_features)

other_features = ['price', 'year']

features_no_tags = [f for f in features if f != 'num_tags']

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def run_model(feature_list):
    X_train_sub = X_train[feature_list]
    X_test_sub = X_test[feature_list]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sub)
    X_test_scaled = scaler.transform(X_test_sub)
    
    model = LogisticRegression(max_iter=3000, class_weight='balanced')
    model.fit(X_train_scaled, y_train)
    
    y_probs = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_probs > 0.35).astype(int)
    
    f1 = f1_score(y_test, y_pred)
    
    return f1, classification_report(y_test, y_pred)

In [ ]:
run_model(dev_features)

In [ ]:
run_model(genre_features)

In [ ]:
run_model(meta_features)

In [ ]:
run_model(dev_features + meta_features)

In [ ]:
run_model(features_no_tags)

In [ ]:
run_model(features)